In [ ]:
cd ..

In [ ]:
import numpy as np 
import pandas as pd 
import requests
import json
import pytz
from datetime import datetime
from multithread_processing.base_job import BaseJob
from utils.logger_utils import get_logger
from databases.mongodb_main import MongoDBMain



In [ ]:
##Filter hot_burn
# with open('data/special_wallets/burn_wallets.json', 'r') as file:
#     burn_data = json.load(file)

# with open('data/special_wallets/hot_wallets.json', 'r') as file:
#     hot_data = json.load(file)

# burn_addresses = []    
# hot_addresses = []    

# for _, burn_value in burn_data.items():
#     burn_addresses.extend(burn_value)
# for _, hot_name in hot_data.items():
#     value_lst = list(hot_name.values())
#     addresses_lst = value_lst[1]
#     for _, hot_address in addresses_lst.items():
#         hot_addresses.extend(hot_address)
# burn_addresses = set(burn_addresses)
# burn_addresses.update(hot_addresses)
# hot_burn_addresses = set(burn_addresses)

In [ ]:
class GetTransactionInfo(BaseJob):
    def __init__(self, chain, collections, historical_token_price, wallets_list: list, filename: str, start_date, end_date,
                 max_workers, batch_size):
        
        mongo = MongoDBMain(chain=chain)
        self.transac_collection = mongo._db[collections]        
        self.transaction_info = list()
        self.transfer_info = list()
        self.filename = filename
        self.start_date = start_date
        self.end_date = end_date
        self.historical_token_price = historical_token_price
        super().__init__(work_iterable=wallets_list,
                         max_workers=max_workers,
                         batch_size=batch_size)

    def _execute_batch(self, works):
        df_list = list()
        cursor =self.transac_collection.find({"block_timestamp" : {"$gte": self.start_date, "$lte": self.end_date},'from_address': {'$in': works}},{'from_address':1, 'to_address':1, 'value':1, 'block_timestamp':1,'transaction_type':1})
        for doc in cursor:
            single_transaction_info = dict(doc) 
            if single_transaction_info['value'] == '0':
                continue
            else:
                round_timestamp = (single_transaction_info['block_timestamp']  - single_transaction_info['block_timestamp']  % 86400) *1000
                # round_timestamp = (single_transaction_info['block_timestamp'] // 3600) * 3600000
                single_transaction_info['valueInUSD'] = int(single_transaction_info['value']) * self.historical_token_price['ethereum'][round_timestamp]
                df_list.append(single_transaction_info)    

        df_concat = pd.DataFrame(df_list)
        self.transaction_info.append(df_concat)
        
    def _end(self):
        super()._end()
        df = pd.concat(self.transaction_info, ignore_index=True)
        #return df
        df.to_csv(self.filename, index=False)


In [ ]:
def format_prices(prices):
    formatted_prices = {}
    for price in prices:
        timestamp = price[0]
        price_value = price[1]
        formatted_prices[timestamp] = price_value
    return formatted_prices

def get_historical_prices(token_id, days=90, vs_currency='usd'):
    url = f'https://api.coingecko.com/api/v3/coins/{token_id}/market_chart?vs_currency={vs_currency}&days={days}'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        prices = data.get('prices', [])
        formatted_prices = format_prices(prices)
        prices_dict = {token_id: formatted_prices}
        return prices_dict
    else:
        return None

In [ ]:
historical_token_price = get_historical_prices('ethereum')
historical_token_round_price = dict()
historical_token_round_price['ethereum'] = {(timestamp - (timestamp % 86400000)): value for timestamp, value in historical_token_price['ethereum'].items()}


In [ ]:
logger = get_logger('User infromation exporter')
input_df = pd.read_json('data/reputation_score/aave_reputation.json')
addresses = input_df.wallet.unique().tolist()[0:5]

In [ ]:

job = GetTransactionInfo(chain='ethereum', 
                            collections='transactions', 
                            historical_token_price=historical_token_round_price,
                            wallets_list= addresses, 
                            filename='data/transaction_info/test.csv', 
                            start_date=1701388800, 
                            end_date= 1708992000, 
                            max_workers=4, 
                            batch_size=10)
job.run()


## test

In [ ]:
import pandas as pd
df = pd.read_csv("/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/transaction_info/test_receiving_groupby.csv")
df

##  Postgres test

In [ ]:
blockchain_etl = MongoDBMain(chain='ethereum')

block_timestamp_mappings = blockchain_etl.get_block_number_to_timestamp(start_block=18687851,
                                                                                                      end_block=19129889)

In [ ]:
from databases.postgresql import PostgresDB
postgres = PostgresDB()
transfers_data = postgres.query_addresses(addresses=addresses,from_block=18687851,
                                                        to_block=19129889,
                                                        chain_id='0x1')
output_data = []
fields_to_select =['contract_address', 'block_number', 'from_address', 'to_address', 'value']
for row in transfers_data:
    new_row = row[:1] + (block_timestamp_mappings[row[1]],) +row[2:]

    output_data.append(dict(zip(fields_to_select, new_row)))

### Mongo Entity

In [ ]:
from databases.mongodb_entity import MongoDBEntity


In [ ]:
input_df = pd.read_json('data/reputation_score_27_2/aave_reputation.json')
input_address = input_df.wallet.unique().tolist()[0:5]
addresses = []
for address in input_address:
    addresses.append('0x1_'+address)


In [ ]:
addresses

In [ ]:
mongo_klg = MongoDBEntity()
mongo_klg_wallet=mongo_klg.wallets
wallets = mongo_klg_wallet.find({'_id': {'$in': addresses}})
wallet_lst = []
for wallet in wallets:
    wallet_lst.append(wallet)
wallet_df = pd.DataFrame(wallet_lst)
wallet_df.columns

In [ ]:
import requests
import time
def format_prices(prices):
    formatted_prices = {}
    for price in prices:
        timestamp = price[0]
        price_value = price[1]
        formatted_prices[timestamp] = price_value
    return formatted_prices

def get_historical_prices(token_id, days=92, vs_currency='usd'):
    url = f'https://api.coingecko.com/api/v3/coins/{token_id}/market_chart?vs_currency={vs_currency}&days={days}'
    response = requests.get(url)
    print(response.status_code)
    if response.status_code == 200:
        time.sleep(3)
    elif response.status_code == 429:
        time.sleep(60)
        response = requests.get(url)
    else:
        return None
    
    data = response.json()
    prices = data.get('prices', [])
    formatted_prices = format_prices(prices)
    prices_dict = {token_id: formatted_prices}
    return prices_dict

In [ ]:
get_historical_prices('bnb')

In [ ]:
aave_from = pd.read_csv("/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/transfer_info/from_transfer.csv")
aave_to = pd.read_csv("/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/transfer_info/to_transfer.csv")


In [ ]:
import json

In [ ]:
token_from = aave_from.contract_address.unique().tolist()
token_to = aave_to.contract_address.unique().tolist()
token_from.extend(token_to)
tokens = list(set(token_from))
with open('/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/tokens/aave_tokens.json', "w") as json_file:
    json.dump(tokens, json_file, indent=4)

In [ ]:
with open('/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/tokens/aave_tokens.json', "r") as json_file:
    data = json.load(json_file)

In [ ]:
historical_token_price = get_historical_prices('airnft-token')
print(historical_token_price)
historical_token_round_price = dict()

historical_token_round_price['airnft-token'] = {(timestamp - (timestamp % 86400000)): value for timestamp, value in historical_token_price['airnft-token'].items()}

In [ ]:

import plotly.plotly as py
from plotly.graph_objs import *
py.sign_in('username', 'api_key')
trace1 = {
  "uid": "bc205455-f2a3-4d3f-abf3-c40c95a88a07", 
  "type": "bar", 
  "x": [0.00013121034475573932, 0.00014993766193974447, 0.0002585219182364357, 0.0010521100636865248, 0.001284185169798048, 0.001329189790355068, 0.0014440240241685514, 0.0018821589336620167, 0.0022027307162027663, 0.002946483536984531, 0.006552406418814974, 0.007849215937398669, 0.009352968392914428, 0.035643426038373754, 0.0616390968690732, 0.09586633897352569, 0.1085875298538611, 0.6618284653562488], 
  "y": ["cuarto_de_servicio", "cerca_parques", "tiempo_de_construido", "habitaciones", "gimnasio", "knn_50_habitaciones_ratio", "bano_hab_ratio", "knn_area_area_ratio", "knn_50_estrato_ratio", "knn_50_area_ratio", "banos", "con_muebles", "knn_area", "knn_pred_geo", "parqueadero", "estrato", "knn_pred_1", "area"], 
  "orientation": "h"
}
data = Data([trace1])
layout = {
  "title": {"text": "Feature Importance"}, 
  "width": 800, 
  "xaxis": {"title": {"text": "Feature Importance"}}, 
  "yaxis": {
    "title": {
      "font": {"size": 16}, 
      "text": "features"
    }, 
    "tickfont": {"size": 10}
  }, 
  "height": 800, 
  "margin": {"l": 200}, 
  "autosize": False
}
fig = Figure(data=data, layout=layout)
plot_url = py.plot(fig)

In [ ]:
len(importances)

In [ ]:

importances = np.array(['Diff2VecSimi', 'XSentNativeToken','XReceivedNativeToken','SubXSentBUSD-T','SubXSentNativeToken','XSentBUSD-T','XReceivedBUSD-T','SubXReceivedNativeToken','SubXReceivedBUSD-T','SubXSentBUSD','XReceivedBUSD','XSentBUSD','SubXReceivedBUSD', 'XTime11','XTime12','XTime13','XTime14','XTime15','XTime10','XTime9'])

# Get feature names
feature_names = np.array([9054,8501,8342,8213,7500,7401,6500,6402,6250,5902,5801,5602,5580,5575,5570,5565,5560,5555,5550,5545])




# Create a bar chart with the specified color scale
fig = go.Figure(data=go.Bar(
            x=importances,
            y=feature_names,
            orientation='h',
            marker=dict(color=importances),
            hovertemplate='%{x:.2f}',
        ))

fig.update_layout(title='Feature Importance',
                  xaxis_title='Importance',
                  yaxis_title='Features')

fig.show()


In [ ]:
import plotly.graph_objects as go

# Your category features list
category_features = ['Diff2VecSimi', 'XSentNativeToken','XReceivedNativeToken','SubXSentBUSD-T','SubXSentNativeToken','XSentBUSD-T','XReceivedBUSD-T','SubXReceivedNativeToken','SubXReceivedBUSD-T','SubXSentBUSD','XReceivedBUSD','XSentBUSD','SubXReceivedBUSD', 'XTime11','XTime12','XTime13','XTime14','XTime15','XTime10','XTime9']

# Their corresponding importance scores
importance_scores = [9054,8501,8342,8213,7500,7401,6500,6402,6250,5902,5801,5602,5580,5575,5570,5565,5560,5555,5550,5545]
# category_features.reverse()
# importance_scores.reverse()

# Create a bar trace
trace = go.Bar(
    y=category_features,  # <-- Set y to category features
    x=importance_scores,  # <-- Set x to importance scores
    orientation='h',  # <-- Set orientation to horizontal
    marker=dict(color='blue'),  # Customize the color if needed
)

# Create layout
layout = go.Layout(
    title='Feature Importance',
    yaxis=dict(title='Features'),  # <-- Set y-axis to features
    xaxis=dict(title='Importance Score'),  # <-- Set x-axis to importance score
)

# Create figure
fig = go.Figure(data=[trace], layout=layout)

# Show the plot
fig.show()

In [ ]:
import plotly.graph_objects as go

category_features = ['Diff2VecSimi', 'XSentBNB','XReceivedBNB','SubXSentBUSD-T','SubXSentBNB','XSentBUSD-T','XReceivedBUSD-T','SubXReceivedBNB','SubXReceivedBUSD-T','SubXSentBUSD','XReceivedBUSD','XSentBUSD','SubXReceivedBUSD', 'XTime11','XTime12','XTime13','XTime14','XTime15','XTime10','XTime9']
importance_scores = [9254,8251,8142,7913,7500,7401,6500,6402,6250,3902,3801,3602,3580,2850,2780,2720,2660,2605,2550,2505]


category_features.reverse()
importance_scores.reverse()
colorscale = [[0, 'rgb(255,209,162)'], [1, 'rgb(197,121,58)']] 

# Create a bar trace
trace = go.Bar(
    y=category_features,  # <-- Set y to category features
    x=importance_scores,  # <-- Set x to importance scores
    orientation='h',  # <-- Set orientation to horizontal
  marker=dict(color=importance_scores, colorscale=colorscale),  # Customize the color if needed
)

# Create layout
layout = go.Layout(
    height=len(category_features)*30,  # Adjust height based on the number of features
    xaxis=dict(tickfont=dict(size=11))  # <-- Set x-axis to importance score
)



# Create figure
fig = go.Figure(data=[trace], layout=layout)
fig.update_layout(


    yaxis={
        'tickfont': {'size': 11}  # Adjust the size as needed
    }
)

fig.show()# Show the plot
fig.write_image("/Users/nguyenhuyhai/20231/Centic/address-clustering-modeling/BNB_feature_importances.svg")


In [ ]:
len(['Diff2VecSimi', 'SubXTimeStd','XTimeStd','SubXTimeMean','XTimeMean','XSentTether','SubXSentTether','XReceivedTether','SubXReceivedTether','XSentUSDC','SubXSentUSDC','XReceivedUSDC','SubXReceivedUSDC', 'XTime11','XTime12','XTime13','XTime14','XTime15','XTime10','XTime9'])

In [ ]:
import plotly.graph_objects as go

category_features = ['Diff2VecSimi', 'SubXTimeStd','XTimeStd','SubXTimeMean','XTimeMean','XSentTether','SubXSentTether','XReceivedTether','SubXReceivedTether','XSentUSDC','SubXSentUSDC','SubXReceivedUSDC', 'XSentETH','XReceivedUSDC','SubXSentETH','XReceivedETH','SubXReceivedETH','XTime15','XTime10','XTime9']

importance_scores = [9254,8891,8842,7783,7500,7401,6003,5802,5750,3902,3801,3602,3580,3350,2780,2720,2460,2305,2250,2205]

category_features.reverse()
importance_scores.reverse()
colorscale = [[0, 'rgb(255,209,162)'], [1, 'rgb(197,121,58)']] 

# Create a bar trace
trace = go.Bar(
    y=category_features,  # <-- Set y to category features
    x=importance_scores,  # <-- Set x to importance scores
    orientation='h',  # <-- Set orientation to horizontal
  marker=dict(color=importance_scores, colorscale=colorscale),  # Customize the color if needed
)

# Create layout
layout = go.Layout(
    height=len(category_features)*30,  # Adjust height based on the number of features
    xaxis=dict(title=' Mean Decrease in Impurity (MDI)'),  # <-- Set x-axis to importance score
)

fig = go.Figure(data=[trace], layout=layout)

fig.write_image("/Users/nguyenhuyhai/20231/Centic/address-clustering-modeling/ETH_feature_importances.svg")
fig.show()

In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Sample DataFrame
data = {
    'SimiScore': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1],

    'Hamming': [0.0, 0.2, 0.32, 0.43, 0.54, 0.55, 0.67, 0.71, 0.75, 0.9, 1],
    'Jaro-Winkler':  [0.0, 0.2, 0.32, 0.43, 0.51, 0.65, 0.75, 0.9, 0.92, 0.95, 1],

    'Levenshtein':  [0.0, 0.1, 0.15, 0.33, 0.44, 0.55, 0.6, 0.62, 0.73, 0.85, 1],
    'Jaro':  [0.0, 0.2, 0.37, 0.46, 0.52, 0.57, 0.65, 0.72, 0.82, 0.88, 1],
    'Prefix': [0.0, 0.1, 0.25, 0.3, 0.42, 0.5, 0.57, 0.7, 0.73, 0.85, 1],
    'Post': [0.0, 0.2, 0.22, 0.25, 0.44, 0.55, 0.67, 0.72, 0.78, 0.82, 1]
}

df = pd.DataFrame(data)

# Create the line chart
fig = go.Figure()

for column in df.columns[1:]:  # Exclude the 'Year' column
    fig.add_trace(go.Scatter(x=df['SimiScore'], y=df[column], mode='lines+markers', name=column))

# Update chart layout
fig.update_layout(xaxis_title='Similarity score',
                  yaxis_title='Identical Proportion')

# Show the chart
fig.show()
fig.write_image("/Users/nguyenhuyhai/20231/Centic/address-clustering-modeling/IdenticalProportion.svg")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
# Assuming your dataframe is named 'df' and contains all the necessary data

data = {
    'SimiScore': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1],

    'Hamming': [0.0, 0.2, 0.32, 0.43, 0.54, 0.55, 0.67, 0.71, 0.75, 0.9, 1],
    'Jaro-Winkler':  [0.0, 0.2, 0.32, 0.43, 0.51, 0.65, 0.75, 0.9, 0.92, 0.95, 1],

    'Levenshtein':  [0.0, 0.1, 0.15, 0.33, 0.44, 0.55, 0.6, 0.62, 0.73, 0.85, 1],
    'Jaro':  [0.0, 0.2, 0.37, 0.46, 0.52, 0.57, 0.65, 0.72, 0.82, 0.88, 1],
    'Prefix': [0.0, 0.1, 0.25, 0.3, 0.42, 0.5, 0.57, 0.7, 0.73, 0.85, 1],
    'Post': [0.0, 0.2, 0.22, 0.25, 0.44, 0.55, 0.67, 0.72, 0.78, 0.82, 1]
}

df = pd.DataFrame(data)

x_values = df['SimiScore']

# Get the list of column names (excluding the x-axis column)
columns_to_plot = df.columns.tolist()
columns_to_plot.remove('SimiScore')

# Plot each column
for column in columns_to_plot:
    # Get y values for the current column
    y_values = df[column]
    
    # Plot the line chart with data points
    plt.plot(x_values, y_values, marker='.', label=column)

# Adding labels and title
plt.xlabel('Similarity score')
plt.ylabel('Identical Proportion')
plt.title('Line Chart Title')

# Add legend
plt.legend()

# Display the plot
plt.grid(True)  # Add grid lines for better visualization
plt.savefig("/Users/nguyenhuyhai/20231/Centic/address-clustering-modeling/string simi.svg", format='svg')

# Show the plot
plt.show()

In [ ]:
cd ..

In [ ]:
from databases.arangodb_klg import ArangoDB

In [ ]:
arangodb =ArangoDB()

In [ ]:
addr = arangodb.get_num_send_receive('0x6f98f8293073b8017422d3afa8b35a34f4ebd375')

In [ ]:
addr['numberSent']

In [ ]:

# Define your AQL query with conditions
aql_query = '''
    FOR doc IN ethereum_addresses
    FILTER doc._id == @value && doc.numberSent<= 1000 && doc.numberReceived <=1000
    RETURN doc
'''

bind_vars = {'value': 'ethereum_addresses/0x1_0x6f98f8293073b8017422d3afa8b35a34f4ebd375'}
cursor = arangodb._db.aql.execute(aql_query, bind_vars=bind_vars)




In [ ]:
[doc for doc in cursor][0]

In [ ]:
cd ..

In [ ]:
from pymongo import MongoClient
from databases.mongodb import MongoDB

In [ ]:
mongoDb = MongoDB()
cursor = mongoDb._db['available_addresses'].find({},{'address':1})

In [ ]:
addresses = []
for cur in cursor[0:10]:
    addresses.append(cur['address'])
addresses

In [ ]:
cd ..

In [ ]:
my_dict = {"key1": 123, "key2": 456}
"key12" not in my_dict

In [ ]:
from databases.mongodb_main import MongoDBMain

In [ ]:
mongooo = MongoDBMain(chain='ethereum')
abcd  = mongooo.get_block_number_to_timestamp(start_block=18809167, end_block=19457983)



In [ ]:
import json
with open('/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis/data/special_wallets/hot_wallets.json', 'r') as file:
    hot_wallets = json.load(file)


In [ ]:
hot_wallets

In [ ]:
pwd

In [ ]:
cd ..

In [ ]:
from services.get_history_price_service import get_historical_prices
eth_price = get_historical_prices('ethereum', days = 98)

In [ ]:
import json
import pandas as pd

with open('data/searching_info_18_3/aave_tokens.json','r') as file:
    token_history = json.load(file)

In [ ]:
token_history.update(eth_price)

In [ ]:
with open('data/testing_data_18_3/aave_tokens.json','w') as file:
    json.dump(token_history, file, indent=4)

In [ ]:
cd ..

In [ ]:
import json
import pandas as pd

with open('data/testing_data_18_3/aave_tokens.json','r') as file:
    token_history = json.load(file)

In [ ]:
token_history['ethereum']

In [ ]:
cd ..

In [ ]:
from databases.mongodb import MongoDB

In [ ]:
mongoo = MongoDB()

In [ ]:
mongoo_avai = mongoo._db['searching_addresses_filter_v2']

In [ ]:

cursor= mongoo_avai.find()

In [ ]:
for c in cursor[0:6]:
    print(c['_id'])

In [ ]:
max_value = 10
range_value = 3

index_list = list(range(range_value, max_value, range_value))
index_list.append(max_value)  # Ensure max_value is included in the list

print(index_list)



In [ ]:
length = 5
range_value = 4

index_list = [1 + i * range_value for i in range(length)]

print(index_list)


In [ ]:
def rename_key(list_of_dicts, old_key, new_key):
    return [{new_key if k == old_key else k: v for k, v in d.items()} for d in list_of_dicts]


# Example usage
list_of_dicts = [
    {'name': 'John', 'age': 30},
    {'name': 'Alice', 'age': 25}
]

list_of_dicts = rename_key(list_of_dicts, 'name', 'full_name')

print(list_of_dicts)


In [1]:
cd ..

/Users/nguyenhuyhai/20232/Centic/RecSys/reputation-analysis


In [2]:
import json
import pandas as pd
import numpy as np
from databases.mongodb import MongoDB

In [3]:
mongodb = MongoDB()
mongo_searching_18_3 = mongodb._db['addresses_searching_18_3']
cursor = mongo_searching_18_3.find({'tags':{ '$in': ['aave_users', 'aave_v3_borrowers', 'aave_v2_borrowers', 'aave_v2_depositors', 'aave_v3_borrowers', 'aave_v3_depositors','stkaave_holders']}},{'_id':1,'tags':1})

In [4]:
with open('data/aave_data_24_3/aave_addresses.json','r') as file:
    aave_address = json.load(file)

address_to_exclude ='0x1_0x0000000000000000000000000000000000000001'
if address_to_exclude in aave_address:
    aave_address.remove(address_to_exclude)


In [5]:
from services.generate_training_data_service import get_value_features, get_freq_features, get_balance_features, get_df

In [10]:
cursor = mongo_searching_18_3.find({'_id':{ '$in': aave_address}},{'_id':1,'interactedWallet':1,'tokenChangeLogs':1,'balanceChangeLogs':1,'depositChangeLogs':1,'borrowChangeLogs':1})
transact_lst = ['_id','interactedWallet']
balance_lst = ['_id','balanceChangeLogs','depositChangeLogs','borrowChangeLogs','tokenChangeLogs']
info_df_lst = []
for aave_users in cursor[0:10]:
    aave_transac = {key: aave_users.get(key) for key in transact_lst}
    aave_balance =  {key: aave_users.get(key) for key in balance_lst}
    transac_df = get_df(aave_transac)
    value_df = get_value_features(transac_df)
    freq_df = get_freq_features(transac_df)
    balance_df = get_balance_features(pd.DataFrame([aave_balance]))
    info_df = value_df.merge(freq_df, on='_id',how = 'inner')
    info_df = info_df.merge(balance_df, on='_id', how='inner')
    info_df_lst.append(info_df)
   # aave_users_info.append(get_df(aave_users))


In [11]:
info_df_lst[0]

,_id,totalUSDTransfer,avgUSDTransfer,totalInUSD_0x0000000000000000000000000000000000000000,totalInUSD_0x030ba81f1c18d280636f32af80b9aad02cf0854e,totalInUSD_0x059956483753947536204e89bfad909e1a434cc6,totalInUSD_0x0f5d2fb29fb7d3cfee444a200298f468908cc942,totalInUSD_0x1a4b46696b2bb4794eb3d4c26f1c55f9170fa4c5,totalInUSD_0x243cacb4d5ff6814ad668c3e225246efa886ad5a,totalInUSD_0x549020a9cb845220d66d3e9c6d9f9ef61c981102,...,totalInUSD_0xa47c8bf37f92abed4a126bda807a7b7498661acd,totalInUSD_0xc18360217d8f7ab5e7c516566761ea12ce7f9d72,numInteractedToken,numInteractedWallet,addressTimeStd,tokenTimeStd,noTransaction,tokenBalanceStd,overallBalance,reputationScore
0,0x1_0x00000000001adc2c0b202d0f72ad9d50f0675296,212.444417,8.131608,145.446136,11.516345,0,5.268702,0,0,7.120183,...,19.916325,8.131608,11,2,5.5,0.57496,13,33.877561,139.954376,2.187302


In [ ]:
balance_df = pd.DataFrame([aave_balance])
balance_df = get_balance_features(balance_df)

In [ ]:
balance_df

In [ ]:
df_freq = get_freq_features(df)

In [ ]:
df_value = get_value_features(df)

In [ ]:
df_freq

In [ ]:
df_value

In [ ]:
df